# 01 Data Loading

This notebook validates the raw-data integration phase for the four approved dataset categories only. It checks the environment, confirms approved raw folders, lists supported and unsupported raw files, previews discovered datasets safely, and regenerates inventory reports without starting EDA, preprocessing, HWI scoring, or model training.


In [7]:
from __future__ import annotations

import platform
import sys
from pathlib import Path

def detect_project_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not detect the project root.")

PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")


Project root: C:\DBS\Sem 2\Aplied Research\HWI
Python executable: C:\DBS\Sem 2\Aplied Research\HWI\venv\Scripts\python.exe
Python version: 3.13.5
Platform: Windows-11-10.0.26200-SP0


In [3]:
import pandas as pd
from IPython.display import display

from src.config import (
    APPROVED_RAW_DATASET_CATEGORIES,
    DATASET_METADATA_PATH,
    RAW_DATA_DIR,
    REPORTS_DIR,
)
from src.data_loader import (
    build_dataset_inventory,
    create_raw_file_manifest,
    discover_dataset_files,
    find_unexpected_raw_categories,
    load_dataset_metadata,
    load_dataset_preview,
    resolve_dataset_metadata,
    validate_approved_raw_categories,
)

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Reports directory: {REPORTS_DIR}")
print(f"Approved raw categories: {APPROVED_RAW_DATASET_CATEGORIES}")
print(f"Dataset metadata file: {DATASET_METADATA_PATH}")


Raw data directory: C:\DBS\Sem 2\Aplied Research\HWI\data\raw
Reports directory: C:\DBS\Sem 2\Aplied Research\HWI\outputs\reports
Approved raw categories: ('awareness', 'ai_emails', 'emails', 'phishing_urls')
Dataset metadata file: C:\DBS\Sem 2\Aplied Research\HWI\references\dataset_metadata.json


In [ ]:
category_status = validate_approved_raw_categories(RAW_DATA_DIR)
unexpected_categories = [path.name for path in find_unexpected_raw_categories(RAW_DATA_DIR)]
manifest_df = create_raw_file_manifest(RAW_DATA_DIR)
supported_manifest_df = manifest_df[manifest_df["supported"] == True].copy() if not manifest_df.empty else pd.DataFrame()
unsupported_manifest_df = manifest_df[manifest_df["supported"] == False].copy() if not manifest_df.empty else pd.DataFrame()

display(pd.DataFrame([{"category": key, "exists": value} for key, value in category_status.items()]))
print(f"Unexpected raw categories: {unexpected_categories if unexpected_categories else "None"}")
print(f"Supported dataset files discovered: {len(supported_manifest_df)}")
print(f"Unsupported raw files discovered: {len(unsupported_manifest_df)}")

if not supported_manifest_df.empty:
    display(supported_manifest_df)
if not unsupported_manifest_df.empty:
    display(unsupported_manifest_df)


In [ ]:
metadata_map = load_dataset_metadata(DATASET_METADATA_PATH)
preview_rows = []
preview_failures = []

for dataset_path in discover_dataset_files(RAW_DATA_DIR):
    relative_path = dataset_path.relative_to(RAW_DATA_DIR).as_posix()
    metadata = resolve_dataset_metadata(dataset_path, metadata_map, raw_data_dir=RAW_DATA_DIR)
    try:
        preview_df = load_dataset_preview(dataset_path, preview_rows=5)
        preview_rows.append(
            {
                "relative_path": relative_path,
                "category": relative_path.split("/")[0],
                "extension": dataset_path.suffix.lower(),
                "size_mb": round(dataset_path.stat().st_size / (1024 * 1024), 4),
                "preview_row_count": len(preview_df),
                "column_names": list(preview_df.columns),
                "dtypes": {str(column): str(dtype) for column, dtype in preview_df.dtypes.items()},
                "candidate_target_suggestions": [
                    column
                    for column in preview_df.columns
                    if any(keyword in str(column).lower() for keyword in ("label", "target", "class", "type", "outcome", "phishing"))
                ],
                "unit_of_analysis_note": metadata.get("unit_of_analysis", "Requires verification"),
            }
        )
    except Exception as exc:
        preview_failures.append(
            {
                "relative_path": relative_path,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

if preview_rows:
    display(pd.DataFrame(preview_rows))
if preview_failures:
    print("Preview failures:")
    display(pd.DataFrame(preview_failures))


In [8]:
inventory_df, quality_df, diagnostics = build_dataset_inventory(
    raw_data_dir=RAW_DATA_DIR,
    metadata_path=DATASET_METADATA_PATH,
    output_dir=REPORTS_DIR,
    return_diagnostics=True,
)

print("Inventory reports written to:")
print(f" - {REPORTS_DIR / "dataset_inventory.csv"}")
print(f" - {REPORTS_DIR / "dataset_inventory.json"}")
print(f" - {REPORTS_DIR / "data_quality_summary.csv"}")
print(f" - {REPORTS_DIR / "raw_file_manifest.csv"}")
print(f" - {REPORTS_DIR / "unsupported_raw_files.csv"}")
print(f"Successful dataset loads: {len(diagnostics.successful_files)}")
print(f"Failed dataset loads: {len(diagnostics.failed_files)}")

display(inventory_df.head())
display(quality_df.head())

if diagnostics.failed_files:
    display(pd.DataFrame([{"relative_path": key, "error": value} for key, value in diagnostics.failed_files.items()]))


2026-07-20 19:28:15,578 | INFO | src.data_loader | Profiling dataset: ai_emails/human-generated/legit.csv
2026-07-20 19:28:15,860 | INFO | src.data_loader | Profiling dataset: ai_emails/human-generated/phishing.csv
2026-07-20 19:28:15,920 | INFO | src.data_loader | Profiling dataset: ai_emails/llm-generated/legit.csv
2026-07-20 19:28:15,973 | INFO | src.data_loader | Profiling dataset: ai_emails/llm-generated/phishing.csv
2026-07-20 19:28:16,002 | ERROR | src.data_loader | Failed to load dataset ai_emails/llm-generated/phishing.csv: DatasetLoaderError: Failed to read CSV dataset 'phishing.csv' using encoding 'utf-8': Error tokenizing data. C error: Expected 7 fields in line 3, saw 13

2026-07-20 19:28:16,007 | INFO | src.data_loader | Profiling dataset: awareness/phishing_awareness_dataset.csv
2026-07-20 19:28:16,153 | INFO | src.data_loader | Profiling dataset: emails/CEAS_08.csv
2026-07-20 19:28:19,643 | INFO | src.data_loader | Profiling dataset: emails/Enron.csv
2026-07-20 19:28:20

Inventory reports written to:
 - C:\DBS\Sem 2\Aplied Research\HWI\outputs\reports\dataset_inventory.csv
 - C:\DBS\Sem 2\Aplied Research\HWI\outputs\reports\dataset_inventory.json
 - C:\DBS\Sem 2\Aplied Research\HWI\outputs\reports\data_quality_summary.csv
 - C:\DBS\Sem 2\Aplied Research\HWI\outputs\reports\raw_file_manifest.csv
 - C:\DBS\Sem 2\Aplied Research\HWI\outputs\reports\unsupported_raw_files.csv
Successful dataset loads: 12
Failed dataset loads: 1


,relative_path,dataset_path,file_name,extension,file_size_bytes,dataset_category,load_status,load_error,row_count,column_count,...,dtypes,missing_value_counts,duplicate_row_count,memory_usage_bytes,candidate_target_columns,unique_value_counts,unit_of_analysis_note,licence_note,metadata_notes,generated_at_utc
0,ai_emails/human-generated/legit.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,legit.csv,.csv,4371139,ai_emails,success,,1000.0,7.0,...,"{'sender': 'str', 'receiver': 'str', 'date': '...","{'sender': 0, 'receiver': 16, 'date': 0, 'subj...",0.0,4408646.0,[label],"{'sender': 345, 'receiver': 199, 'date': 694, ...",Requires verification,Requires verification,,2026-07-20T18:28:15.853474+00:00
1,ai_emails/human-generated/phishing.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,phishing.csv,.csv,1337693,ai_emails,success,,1000.0,7.0,...,"{'sender': 'str', 'receiver': 'str', 'date': '...","{'sender': 0, 'receiver': 22, 'date': 0, 'subj...",496.0,1379879.0,[label],"{'sender': 473, 'receiver': 60, 'date': 504, '...",Requires verification,Requires verification,,2026-07-20T18:28:15.917154+00:00
2,ai_emails/llm-generated/legit.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,legit.csv,.csv,648934,ai_emails,success,,1000.0,2.0,...,"{'text': 'str', 'label': 'int64'}","{'text': 0, 'label': 0}",2.0,659054.0,[label],"{'text': 998, 'label': 1}",Requires verification,Requires verification,,2026-07-20T18:28:15.965535+00:00
3,ai_emails/llm-generated/phishing.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,phishing.csv,.csv,661439,ai_emails,failed,DatasetLoaderError: Failed to read CSV dataset...,NaN,NaN,...,{},{},NaN,NaN,[],{},Requires verification,Requires verification,File requires manual structural verification b...,2026-07-20T18:28:16.005884+00:00
4,awareness/phishing_awareness_dataset.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/awar...,phishing_awareness_dataset.csv,.csv,845184,awareness,success,,5000.0,12.0,...,"{'user_id': 'str', 'email_subject': 'str', 'se...","{'user_id': 0, 'email_subject': 0, 'sender_ema...",0.0,1221162.0,[device_type],"{'user_id': 5000, 'email_subject': 5000, 'send...",Requires verification,Requires verification,,2026-07-20T18:28:16.150492+00:00


,relative_path,dataset_path,dataset_category,load_status,load_error,row_count,column_count,total_missing_values,duplicate_row_count,memory_usage_bytes,candidate_target_columns,unit_of_analysis_note,licence_note,generated_at_utc
0,ai_emails/human-generated/legit.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,ai_emails,success,,1000.0,7.0,16.0,0.0,4408646.0,[label],Requires verification,Requires verification,2026-07-20T18:28:15.853474+00:00
1,ai_emails/human-generated/phishing.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,ai_emails,success,,1000.0,7.0,24.0,496.0,1379879.0,[label],Requires verification,Requires verification,2026-07-20T18:28:15.917154+00:00
2,ai_emails/llm-generated/legit.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,ai_emails,success,,1000.0,2.0,0.0,2.0,659054.0,[label],Requires verification,Requires verification,2026-07-20T18:28:15.965535+00:00
3,ai_emails/llm-generated/phishing.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/ai_e...,ai_emails,failed,DatasetLoaderError: Failed to read CSV dataset...,NaN,NaN,NaN,NaN,NaN,[],Requires verification,Requires verification,2026-07-20T18:28:16.005884+00:00
4,awareness/phishing_awareness_dataset.csv,C:/DBS/Sem 2/Aplied Research/HWI/data/raw/awar...,awareness,success,,5000.0,12.0,0.0,0.0,1221162.0,[device_type],Requires verification,Requires verification,2026-07-20T18:28:16.150492+00:00


,relative_path,error
0,ai_emails/llm-generated/phishing.csv,DatasetLoaderError: Failed to read CSV dataset...
